In [2]:
import pandas as pd
# Import the dataset as dataframe first.
dataset = pd.read_csv("dataset.tsv", sep="\t")
df = pd.DataFrame({'book_content': dataset.summary, 'genre':dataset.genre})

df.head(2)

,book_content,genre
0,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,young_adult
1,Harry Potter's life is miserable. His parents ...,fantasy


In [3]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

ps = PorterStemmer()
stop_words = set(stopwords.words('english'))

# text preprocessing
def preprocess_text(text):
    text = text.lower()
    space_version_text = re.sub(r'[^\w\s]+', ' ', text.lower())
    tokens = word_tokenize(space_version_text)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [ps.stem(word) for word in tokens]
    return ' '.join(tokens)

# Apply preprocessing to each book
df['book_content'] = df['book_content'].apply(preprocess_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/carside/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/carside/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/carside/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Split the data into training and testing sets
train_df = df.iloc[:3000]
test_df = df.iloc[3000:4000]

# Convert text data into TF-IDF weights
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(train_df['book_content'])
X_test = vectorizer.transform(test_df['book_content'])

y_train, y_test = train_df['genre'], test_df['genre']


print(X_train.shape, X_test.shape)

(3000, 20854) (1000, 20854)


### 找出SVM的最佳参数

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

X_train_text = train_df["book_content"]
y_train = train_df["genre"]

X_test_text = test_df["book_content"]
y_test = test_df["genre"]

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LinearSVC(max_iter=5000))
])

param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2, 3],
    "tfidf__max_df": [0.8, 0.9, 1.0],
    "tfidf__sublinear_tf": [False, True],
    "clf__C": np.arange(0.1, 2.6, 0.1),
    "clf__class_weight": [None, "balanced"]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train_text, y_train)

print("Best parameters:")
print(grid.best_params_)

print("Best CV macro-F1:")
print(grid.best_score_)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test_text)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 1800 candidates, totalling 9000 fits


/Users/carside/miniconda3/envs/6713/lib/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Users/carside/miniconda3/envs/6713/lib/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Users/carside/miniconda3/envs/6713/lib/python3.11/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from 

### 找出Bernoulli Naive Bayes的最佳参数alpha

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

X_train_text = train_df["book_content"]
y_train = train_df["genre"]

X_test_text = test_df["book_content"]
y_test = test_df["genre"]

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", BernoulliNB())
])

param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2, 3],
    "tfidf__max_df": [0.8, 0.9, 1.0],
    "tfidf__sublinear_tf": [False, True],
    "clf__alpha": np.arange(0.02, 1.02, 0.04),
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train_text, y_train)

print("Best parameters:")
print(grid.best_params_)

print("Best CV macro-F1:")
print(grid.best_score_)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test_text)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

### 找出 Multinomial Naive Bayes的最佳参数alpha

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

X_train_text = train_df["book_content"]
y_train = train_df["genre"]

X_test_text = test_df["book_content"]
y_test = test_df["genre"]

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB())
])

param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2, 3],
    "tfidf__max_df": [0.8, 0.9, 1.0],
    "tfidf__sublinear_tf": [False, True],
    "clf__alpha": np.arange(0.02, 1.04, 0.04),
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train_text, y_train)

print("Best parameters:")
print(grid.best_params_)

print("Best CV macro-F1:")
print(grid.best_score_)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test_text)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))